# 08 Extension: objective (b), sibling datasets to TRAM test

**Status: NOT RUN ON COLAB.** Generated by `scripts/make_notebooks.py`. Any output cells in the repository copy come from a local CPU smoke run (`SMOKE=True`, 20 samples), never from a real run.

| | |
|---|---|
| Purpose | Train on (b) the sibling sources of `combined_train.csv` only and (c) TRAM + siblings, select on TRAM dev, test once on TRAM test. The sibling data IS in the folder (`source_dataset` = procedure, derived_procedure). |
| Inputs | `Dataset/combined_train.csv`, TRAM dev/test |
| Outputs | `cross_dataset_prep.json`; trained runs under `showcase_runs/08_extension_cross_dataset/` |
| Runtime (T4 GPU, estimate) | matcher: about 4x the TRAM run per configuration and seed (estimate: 1.5-2 h); kill-switch candidate |
| Fills in `thesis_numbers.json` | T6.cross_dataset |

Run the cells top to bottom. Set `SMOKE = True` in the setup cell for a dry run on 20 samples. Every run writes a manifest with input and output hashes; the last cell prints an evidence table read from the saved files.

Pre-registration: `experiments/prereg/11_cross_dataset.md`. Sibling rows that duplicate a TRAM dev or test sentence are removed before training; the count is recorded. Only the matcher is trained here: it is the one component whose label space already covers the sibling labels.

In [ ]:
NOTEBOOK = "08_extension_cross_dataset"
NEEDS_GPU = True      # True: the notebook stops at once if Colab gave no GPU

# ---- setup (same in every notebook) ----
import os, sys, json, ast, time, hashlib, random, platform, subprocess, re
from pathlib import Path
import numpy as np, pandas as pd

SMOKE = os.environ.get("TTPMAP_SMOKE", "0") == "1"      # set SMOKE = True by hand for a 20-sample dry run
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive mount failed:", type(e).__name__, e)
    if not os.path.isdir("/content/drive/MyDrive") and not SMOKE:
        raise RuntimeError("Google Drive is not mounted. The Colab extension for VS Code cannot mount Drive (known limitation of the extension), "
                           "and these notebooks read their inputs from and save their results to Drive. Open this notebook in the browser instead: "
                           "https://colab.research.google.com > File > Upload notebook (or open it from Drive). See notebooks/COLAB_RUNBOOK.md, section 0.")
BASE = Path("/content/drive/MyDrive/KDU/Thesis") if IN_COLAB else Path(os.environ.get("TTPMAP_ROOT", r"E:\Osamas Paper"))
DATA_DIR, CLS_ROOT, MATCH_ROOT, SHAP_DIR = BASE / "Dataset", BASE / "Exp_1_4_run_tram_2_epoc_30", BASE / "run_match_1", BASE / "Exp_3_0_shap"
MATCH2 = BASE / "run_match_2_saved"                      # NEW folder: matcher retrained WITH saved weights (notebook 04)
OUT = (Path(os.environ["TTPMAP_SMOKE_OUT"]) if os.environ.get("TTPMAP_SMOKE_OUT") else BASE / "showcase_runs") / NOTEBOOK
OUT.mkdir(parents=True, exist_ok=True)
SEEDS = (42, 43, 44)
if IN_COLAB and not SMOKE and not (BASE / "Dataset" / "tram_test.tsv").exists():
    _msg = (f"Project files not found under {BASE}. Upload the folder E:\\Osamas Paper\\_showcase\\drive_upload\\KDU into 'My Drive' "
            "(drive.google.com, same Google account as this Colab session), then run this cell again. See notebooks/COLAB_RUNBOOK.md, section 0.")
    if NOTEBOOK == "00_setup_and_verify": print("WARNING:", _msg)
    else: raise RuntimeError(_msg)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    except ImportError:
        pass

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for b in iter(lambda: fh.read(1 << 20), b""): h.update(b)
    return h.hexdigest()

def env_info():
    info = {"python": platform.python_version(), "platform": platform.platform(), "in_colab": IN_COLAB, "smoke": SMOKE}
    try:
        import torch
        info.update(torch=torch.__version__, cuda=torch.cuda.is_available(), gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
    except ImportError:
        pass
    return info

def write_manifest(run_id, system, seed, config, inputs, outputs, metrics, status):
    """One manifest per run: what went in, what came out, with hashes. Read later by the ingest step."""
    m = {"run_id": run_id, "system": system, "seed": seed, "config_hash": hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:16], "config": config,
         "env": {k: str(v) for k, v in env_info().items()}, "input_sha256": {str(p): sha256(p) for p in inputs if Path(p).exists()},
         "output_sha256": {str(p): sha256(p) for p in outputs if Path(p).exists()}, "metrics": {k: float(v) for k, v in metrics.items()},
         "status": "SMOKE" if SMOKE else status, "written": time.strftime("%Y-%m-%dT%H:%M:%S")}
    (OUT / f"manifest_{run_id}.json").write_text(json.dumps(m, indent=1)); return m

print(f"notebook={NOTEBOOK} | SMOKE={SMOKE} | IN_COLAB={IN_COLAB} | BASE={BASE} | OUT={OUT}")
print(env_info())
import importlib.metadata as _md
print("packages:", {p: (_md.version(p) if p in {d.metadata["Name"].lower() for d in _md.distributions() if d.metadata["Name"]} else "not installed")
                    for p in ("torch", "transformers", "scikit-learn", "numpy", "pandas", "shap", "scipy")})
_gpu = env_info().get("cuda", False)
if NEEDS_GPU and not _gpu and not SMOKE:
    raise RuntimeError("No GPU found. In Colab: Runtime > Change runtime type > T4 GPU > Save, then run this cell again. (Set SMOKE = True only for a dry run.)")
if not NEEDS_GPU and not _gpu:
    print("note: no GPU; this notebook runs on CPU in a few minutes")

In [ ]:
# ---- data, label space, gold matrices (same as Exp_1_4 CELL 4 and Exp_2.1 CELL 2) ----
def _labels(x): return sorted({str(t).strip().upper() for t in ast.literal_eval(x)})
def load_split(name):
    p = DATA_DIR / f"tram_{name}.tsv"
    if p.exists():
        df = pd.read_csv(p, sep="\t").rename(columns={"text1": "text"})
    elif name == "train":                                   # local fallback: the tram slice of combined_train.csv is the same data
        df = pd.read_csv(DATA_DIR / "combined_train.csv"); df = df[df["source_dataset"] == "tram"].rename(columns={"text1": "text"})
    else:
        raise FileNotFoundError(p)
    df = df[["text", "labels"]].copy(); df["labels"] = df["labels"].apply(_labels)
    return df.drop_duplicates(subset=["text"]).reset_index(drop=True) if name == "train" else df.reset_index(drop=True)

train, dev, test = (load_split(s) for s in ("train", "dev", "test"))
desc = pd.read_csv(DATA_DIR / "ttp-desc-mappings.csv", sep="\t")
desc["ttp"] = desc["ttp"].str.lower().str.strip()
desc["description"] = desc["description"].fillna("").str.replace(r"\s+", " ", regex=True).str.replace(r"\bcitation:\s*", "", regex=True).str.strip()
desc = desc.drop_duplicates("ttp"); desc = desc[~desc["ttp"].isin({"t1060", "t1107", "t1151", "t1188"})].reset_index(drop=True)
LABELS = [t.upper() for t in desc["ttp"]]; LABEL_TEXTS = desc["description"].tolist(); L = len(LABELS); lab2i = {l: i for i, l in enumerate(LABELS)}
def to_Y(df):
    Y = np.zeros((len(df), L), dtype=bool)
    for i, ys in enumerate(df["labels"]):
        for y in ys: Y[i, lab2i[y]] = True
    return Y
Y_train, Y_dev, Y_test = to_Y(train), to_Y(dev), to_Y(test)
train_count = Y_train.sum(0); SEEN = train_count > 0; seen_idx = np.where(SEEN)[0]
assert (len(train), len(dev), len(test), L, int(SEEN.sum())) == (3468, 604, 725, 733, 180), "unexpected data shape"
TEST_IDS = [f"test-{i:04d}" for i in range(len(test))]; DEV_IDS = [f"dev-{i:04d}" for i in range(len(dev))]
N_SMOKE = 20
print(f"train={len(train)} dev={len(dev)} test={len(test)} labels={L} seen={int(SEEN.sum())}")

In [ ]:
# ---- library cell: generated from src/ttpmap (metrics.py, fusion.py, stats.py); do not edit here ----
import itertools, math
from functools import reduce
from typing import Dict, Iterable, List, Sequence, Tuple, Literal
import numpy as np
from scipy.stats import binomtest
F1Variant = Literal['notebook', 'seen_test', 'all_test']
ORDER = ('matcher', 'svm', 'roberta', 'secbert')
BOOT_N, BOOT_SEED = 10_000, 20261114

def rank_topk(scores: np.ndarray, k: int) -> np.ndarray:
    """Indices of the k best labels per row; deterministic under ties."""
    return np.argsort(-scores, axis=1, kind="stable")[:, :k]


def hit_at_1(scores: np.ndarray, Y: np.ndarray) -> np.ndarray:
    """Per-sentence 0/1 vector: is the top-1 label a gold label? Mean of it = P@1."""
    top = rank_topk(scores, 1)[:, 0]
    return Y[np.arange(len(Y)), top].astype(float)


def precision_at_k(scores: np.ndarray, Y: np.ndarray, k: int) -> np.ndarray:
    """Per-sentence (number of gold labels in the top k) / k."""
    return np.take_along_axis(Y, rank_topk(scores, k), axis=1).sum(1) / float(k)


def rr_at_k(scores: np.ndarray, Y: np.ndarray, k: int = 3) -> np.ndarray:
    """Per-sentence reciprocal rank of the first gold label inside the top k; 0 if none. Mean = MRR@k."""
    hits = np.take_along_axis(Y, rank_topk(scores, k), axis=1)
    first = hits.argmax(1) + 1
    return np.where(hits.any(1), 1.0 / first, 0.0)


def recall_at_k_micro(scores: np.ndarray, Y: np.ndarray, k: int) -> float:
    """Gold (sentence, label) pairs found in the top k / all gold pairs (the notebooks' "R@k")."""
    return float(np.take_along_axis(Y, rank_topk(scores, k), axis=1).sum() / Y.sum())


def recall_at_k_per_sample(scores: np.ndarray, Y: np.ndarray, k: int) -> float:
    """Mean over sentences of hits/|gold| (the definition printed in Nguyen et al., App. D.1)."""
    hits = np.take_along_axis(Y, rank_topk(scores, k), axis=1).sum(1)
    return float((hits / np.maximum(Y.sum(1), 1)).mean())


def apply_threshold(scores: np.ndarray, thr: float) -> np.ndarray:
    """Multi-label decision: score >= thr; a row with no label gets its arg-max label (never predict nothing)."""
    P = scores >= thr
    empty = P.sum(1) == 0
    P[np.where(empty)[0], scores[empty].argmax(1)] = True
    return P


def _prf(P: np.ndarray, Y: np.ndarray) -> Dict[str, float]:
    tp = int((P & Y).sum()); fp = int((P & ~Y).sum()); fn = int((~P & Y).sum())
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return {"precision": p, "recall": r, "f1": 2 * p * r / (p + r) if p + r else 0.0}


def f1_per_label(P: np.ndarray, Y: np.ndarray) -> np.ndarray:
    """Label-wise F1 with zero_division = 0."""
    tp = (P & Y).sum(0).astype(float); fp = (P & ~Y).sum(0); fn = (~P & Y).sum(0)
    den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)


def f1_scores(P: np.ndarray, Y: np.ndarray, seen: np.ndarray, variant: F1Variant) -> Dict[str, float]:
    """Micro and macro F1 in one of three explicit variants.

    P, Y: bool (n, n_labels) in the FULL label space; ``seen``: bool (n_labels,) = label occurs in training.

    * ``notebook``  : only the columns of labels seen in training are scored. Gold labels never seen in training are
      ignored (not counted as misses). Macro over seen labels that occur in this split. This is what the original
      notebooks report.
    * ``seen_test`` : micro counts unseen gold labels as false negatives; macro over seen labels that occur in this split.
    * ``all_test``  : micro as in ``seen_test``; macro over ALL labels that occur in this split (unseen labels score 0).
    """
    present = Y.sum(0) > 0
    f = f1_per_label(P, Y)
    if variant == "notebook":
        m = _prf(P[:, seen], Y[:, seen]); macro = f[present & seen].mean()
    elif variant == "seen_test":
        m = _prf(P, Y); macro = f[present & seen].mean()
    elif variant == "all_test":
        m = _prf(P, Y); macro = f[present].mean()
    else:
        raise ValueError(variant)
    return {"micro_f1": m["f1"], "micro_precision": m["precision"], "micro_recall": m["recall"], "macro_f1": float(macro),
            "n_macro_labels": int((present & seen).sum() if variant != "all_test" else present.sum())}


class LeakageError(RuntimeError):
    """Raised when a tuning function is handed anything other than the dev split."""


def to_full(scores: np.ndarray, cols: np.ndarray, L: int, fill: float = np.nan) -> np.ndarray:
    F = np.full((scores.shape[0], L), fill); F[:, cols] = scores
    return F


def zrow(F: np.ndarray) -> np.ndarray:
    """Per-row z-score over the non-NaN labels; NaN (unknown label) -> 0."""
    m = np.nanmean(F, axis=1, keepdims=True); s = np.nanstd(F, axis=1, keepdims=True) + 1e-9
    return np.nan_to_num((F - m) / s, nan=0.0)


def fuse(Z: Dict[str, np.ndarray], w: Sequence[float]) -> np.ndarray:
    return sum(wi * Z[n] for wi, n in zip(w, ORDER))


def coarse_grid() -> List[Tuple[int, ...]]:
    """{0,1,2,3}^4 without the all-zero setting: 255 settings (Exp_2.1 CELL 8)."""
    return [w for w in itertools.product([0, 1, 2, 3], repeat=4) if sum(w)]


def wide_grid() -> List[Tuple[int, ...]]:
    """{0,1,2,3,4,6,8}^4 reduced by gcd: 1,711 distinct settings (Exp_2.1 CELL 9)."""
    seen, out = set(), []
    for w in itertools.product([0, 1, 2, 3, 4, 6, 8], repeat=4):
        g = reduce(math.gcd, w)
        if g == 0:
            continue
        w = tuple(x // g for x in w)
        if w not in seen:
            seen.add(w); out.append(w)
    return out


def select_weights(Z_dev: Dict[str, np.ndarray], Y_dev: np.ndarray, grid: Iterable[Tuple[int, ...]], *, split: str,
                   sentence_ids: Sequence[str] | None = None, forbidden_ids: Sequence[str] = ()) -> Tuple[Tuple[int, ...], Tuple[float, float]]:
    """Pick fusion weights. Objective: DEV MRR@3 (4 d.p.), tie-break DEV P@1; first strictly better setting wins.

    Guard: ``split`` must be the string "dev", and if ``sentence_ids`` are given none may be in ``forbidden_ids``
    (pass the test-set ids there). Violations raise LeakageError.
    """
    if split != "dev":
        raise LeakageError(f"weights may be tuned on dev only, got split={split!r}")
    if sentence_ids is not None and set(sentence_ids) & set(forbidden_ids):
        raise LeakageError("test sentence ids reached a tuning function")
    best = None
    for w in grid:
        Sc = fuse(Z_dev, w)
        key = (round(float(rr_at_k(Sc, Y_dev, 3).mean()), 4), round(float(hit_at_1(Sc, Y_dev).mean()), 4))
        if best is None or key > best[0]:
            best = (key, tuple(w))
    return best[1], best[0]


def select_threshold(scores_dev: np.ndarray, Y_dev: np.ndarray, grid: Sequence[float], *, split: str) -> float:
    """Global decision threshold maximising DEV micro-F1 over the model's own label columns (first maximum wins)."""
    if split != "dev":
        raise LeakageError(f"thresholds may be tuned on dev only, got split={split!r}")
    def mf(t):
        P = scores_dev >= t
        tp = (P & Y_dev).sum(); d = 2 * tp + (P & ~Y_dev).sum() + (~P & Y_dev).sum()
        return 2.0 * tp / d if d else 0.0
    return float(grid[int(np.argmax([mf(t) for t in grid]))])


def bootstrap_ci(x: np.ndarray, n_boot: int = BOOT_N, seed: int = BOOT_SEED, level: float = 0.95) -> Dict[str, float]:
    """Percentile CI of the mean of a per-sentence vector (unit of resampling = sentence)."""
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return {"mean": np.nan, "lo": np.nan, "hi": np.nan, "n": 0}
    rng = np.random.default_rng(seed)
    means = x[rng.integers(0, len(x), (n_boot, len(x)))].mean(1)
    a = (1 - level) / 2
    return {"mean": float(x.mean()), "lo": float(np.quantile(means, a)), "hi": float(np.quantile(means, 1 - a)), "n": int(len(x))}


def paired_bootstrap(a: np.ndarray, b: np.ndarray, n_boot: int = BOOT_N, seed: int = BOOT_SEED) -> Dict[str, float]:
    """Paired bootstrap of mean(a - b) with a percentile 95% CI and a two-sided bootstrap p-value."""
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    rng = np.random.default_rng(seed)
    bs = d[rng.integers(0, len(d), (n_boot, len(d)))].mean(1)
    return {"delta": float(d.mean()), "lo": float(np.quantile(bs, 0.025)), "hi": float(np.quantile(bs, 0.975)),
            "p_boot": float(min(1.0, 2 * min((bs <= 0).mean(), (bs >= 0).mean())))}


def mcnemar_exact(a: np.ndarray, b: np.ndarray) -> Dict[str, float]:
    """Exact two-sided McNemar on two 0/1 hit vectors. b_ = a right & b wrong, c_ = a wrong & b right."""
    a = np.asarray(a).astype(bool); b = np.asarray(b).astype(bool)
    b_ = int((a & ~b).sum()); c_ = int((~a & b).sum())
    p = binomtest(b_, b_ + c_, 0.5).pvalue if b_ + c_ else 1.0
    return {"b": b_, "c": c_, "p": float(p)}


def holm(pvalues: Sequence[float]) -> List[float]:
    """Holm step-down adjusted p-values, returned in the input order."""
    p = np.asarray(pvalues, dtype=float); m = len(p)
    order = np.argsort(p); adj = np.empty(m); running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * p[i]); adj[i] = min(1.0, running)
    return adj.tolist()

In [ ]:
# ---- bi-encoder matcher: model, training WITH saved weights (Exp_2.1 CELL 3 settings) ----
import math, torch, torch.nn as nn, torch.nn.functional as Fnn
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE_MODEL = "BAAI/bge-base-en-v1.5"

class Matcher(nn.Module):
    def __init__(self, name):
        super().__init__(); self.enc = AutoModel.from_pretrained(name)
    def forward(self, **enc):
        return Fnn.normalize(self.enc(**enc).last_hidden_state[:, 0].float(), dim=-1)

def tok_batch(tok, texts, max_len):
    enc = tok(list(texts), padding=True, truncation=True, max_length=max_len, return_tensors="pt")
    return {k: v.to(device) for k, v in enc.items() if k in ("input_ids", "attention_mask", "token_type_ids")}

@torch.no_grad()
def embed_all(model, tok, texts, max_len, bs=64, amp=True):
    model.eval(); out = []
    for i in range(0, len(texts), bs):
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp and device.type == "cuda"):
            out.append(model(**tok_batch(tok, texts[i:i + bs], max_len)).float().cpu())
    return torch.cat(out).numpy()

def matcher_metrics(S, Y):
    return {"P@1": float(hit_at_1(S, Y).mean()), "MRR@3": float(rr_at_k(S, Y, 3).mean()), "R@5": recall_at_k_micro(S, Y, 5)}

def train_matcher_save(out_dir, seed, epochs=20, patience=4, batch_size=16, lr=2e-5, n_neg=64, scale=30.0, text_max_len=128, label_max_len_train=128, label_max_len_eval=256,
                       tr=None, Ytr=None, dv=None, Ydv=None, te=None, Yte=None):
    """Resumable per epoch: best weights -> matcher_state.pt ; last state -> ckpt_last.pt (both on Drive)."""
    tr = train if tr is None else tr; Ytr = Y_train if Ytr is None else Ytr; dv = dev if dv is None else dv; Ydv = Y_dev if Ydv is None else Ydv; te = test if te is None else te; Yte = Y_test if Yte is None else Yte
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True); set_seed(seed); rng = np.random.default_rng(seed); amp_on = device.type == "cuda"
    tok = AutoTokenizer.from_pretrained(BASE_MODEL); model = Matcher(BASE_MODEL).to(device)
    N = len(tr); texts = tr["text"].tolist(); label_texts = np.array(LABEL_TEXTS, dtype=object); pos_lists = [np.where(Ytr[i])[0] for i in range(N)]; pool = np.arange(L)
    steps = math.ceil(N / batch_size); opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps * epochs), steps * epochs); scaler = torch.amp.GradScaler(enabled=amp_on)
    st = {"epoch": 0, "best": -1.0, "best_ep": 0, "bad": 0, "hist": []}; ck = out_dir / "ckpt_last.pt"
    if ck.exists():
        c = torch.load(ck, map_location="cpu", weights_only=False); model.load_state_dict(c["model"]); opt.load_state_dict(c["opt"]); sched.load_state_dict(c["sched"])
        rng.bit_generator.state = c["rng"]; st = c["state"]; print("  resumed after epoch", st["epoch"])
    def evaluate():
        le = embed_all(model, tok, LABEL_TEXTS, label_max_len_eval)
        Sd = embed_all(model, tok, list(dv["text"]), text_max_len) @ le.T; St = embed_all(model, tok, list(te["text"]), text_max_len) @ le.T
        return Sd, St, matcher_metrics(Sd, Ydv), matcher_metrics(St, Yte)
    for ep in range(st["epoch"] + 1, epochs + 1):
        if st["bad"] >= patience: break
        model.train(); perm = rng.permutation(N); tot = 0.0; t0 = time.time()
        for b in range(0, N, batch_size):
            idx = perm[b:b + batch_size]; gold_u = np.unique(np.concatenate([pos_lists[i] for i in idx])); cand = np.setdiff1d(pool, gold_u)
            U = np.concatenate([gold_u, rng.choice(cand, size=min(n_neg, len(cand)), replace=False)])
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_on):
                et = model(**tok_batch(tok, [texts[i] for i in idx], text_max_len)); eu = model(**tok_batch(tok, label_texts[U].tolist(), label_max_len_train))
            S = (et @ eu.T) * scale; pos = torch.tensor(Ytr[idx][:, U], device=device)
            neg_lse = torch.logsumexp(S.masked_fill(pos, -1e4), dim=1); rows = pos.nonzero()[:, 0]
            loss = Fnn.softplus(neg_lse[rows] - S[pos]).mean()
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); tot += loss.item() * len(idx)
        Sd, St, md, mt = evaluate(); st["hist"].append({"epoch": ep, "train_loss": tot / N, **{f"dev_{k}": v for k, v in md.items()}})
        print(f"  epoch {ep:2d}: loss={tot / N:.4f} dev P@1={md['P@1']:.4f} MRR@3={md['MRR@3']:.4f} ({time.time() - t0:.0f}s)")
        if md["MRR@3"] > st["best"]:
            st.update(best=md["MRR@3"], best_ep=ep, bad=0)
            torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, out_dir / "matcher_state.pt")
            np.save(out_dir / "dev_scores.npy", Sd); np.save(out_dir / "test_scores.npy", St)
            cfg = dict(epochs=epochs, patience=patience, batch_size=batch_size, lr=lr, n_neg=n_neg, scale=scale, text_max_len=text_max_len, label_max_len_train=label_max_len_train, pooling="cls", neg_pool="all")
            json.dump({"model_name": BASE_MODEL, "seed": seed, "best_epoch": ep, "dev": md, "test": mt, "history": st["hist"], "config": cfg, "finished": False}, open(out_dir / "metrics.json", "w"), indent=1)
        else:
            st["bad"] += 1
        st["epoch"] = ep
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(), "rng": rng.bit_generator.state, "state": st}, ck)
    m = json.load(open(out_dir / "metrics.json")); m["finished"] = True; m["history"] = st["hist"]; json.dump(m, open(out_dir / "metrics.json", "w"), indent=1)
    del model, opt; import gc; gc.collect(); torch.cuda.empty_cache()
    return m

In [ ]:
comb = pd.read_csv(DATA_DIR / "combined_train.csv").rename(columns={"text1": "text"}); comb["labels"] = comb["labels"].apply(_labels)
norm = lambda s: re.sub(r"\s+", " ", str(s).lower()).strip()
heldout = set(map(norm, dev["text"])) | set(map(norm, test["text"]))
sib = comb[comb["source_dataset"] != "tram"].drop_duplicates(subset=["text"]).reset_index(drop=True)
leak = sib["text"].map(norm).isin(heldout); in_space = sib["labels"].apply(lambda ys: all(y in lab2i for y in ys))
sib_clean = sib[~leak & in_space].reset_index(drop=True)
prep = {"sibling_rows": len(sib), "removed_duplicates_of_dev_or_test": int(leak.sum()), "removed_labels_outside_space": int((~in_space & ~leak).sum()), "sibling_rows_used": len(sib_clean),
        "sibling_labels": len({y for ys in sib_clean["labels"] for y in ys}), "test_labels_covered_by_siblings": int(sum(l in {y for ys in sib_clean["labels"] for y in ys} for l in np.array(LABELS)[Y_test.sum(0) > 0])),
        "unseen_test_labels_covered_by_siblings": int(sum(l in {y for ys in sib_clean["labels"] for y in ys} for l in np.array(LABELS)[(Y_test.sum(0) > 0) & ~SEEN]))}
(OUT / "cross_dataset_prep.json").write_text(json.dumps({"metrics": prep}, indent=1)); print(prep)

In [ ]:
CONFIGS = {"b_siblings_only": sib_clean[["text", "labels"]], "c_tram_plus_siblings": pd.concat([train[["text", "labels"]], sib_clean[["text", "labels"]]], ignore_index=True).drop_duplicates(subset=["text"]).reset_index(drop=True)}
if SMOKE and device.type != "cuda":
    print("SMOKE on CPU: training skipped (GPU-only). Data preparation above ran in full.")
else:
    for name, df in CONFIGS.items():
        for sd in ((42,) if SMOKE else SEEDS):
            out = OUT / f"matcher_{name}_s{sd}"
            if (out / "metrics.json").exists() and json.load(open(out / "metrics.json")).get("finished"): print("skip", out); continue
            d = df.iloc[:N_SMOKE] if SMOKE else df
            m = train_matcher_save(out, sd, epochs=1 if SMOKE else 20, tr=d, Ytr=to_Y(d))
            St = np.load(out / "test_scores.npy"); rarest = np.where(Y_test, train_count[None, :], 10**9).min(1)
            met = {**m["test"], "P@1_unseen_in_tram": float(hit_at_1(St, Y_test)[rarest == 0].mean()), "P@1_support_1_4": float(hit_at_1(St, Y_test)[(rarest >= 1) & (rarest <= 4)].mean()), "train_rows": len(d)}
            write_manifest(f"matcher_{name}_s{sd}", f"matcher-{name}", sd, m["config"], [DATA_DIR / "combined_train.csv"], [out / "test_scores.npy", out / "metrics.json"], met, "DONE"); print(name, sd, met)

In [ ]:
# ---- evidence table: read back from the files this notebook wrote (no typed numbers) ----
rows = []
for p in sorted(OUT.glob("*.json")):
    d = json.loads(p.read_text())
    flat = d.get("metrics", d) if isinstance(d, dict) else {}
    for k, v in flat.items():
        if isinstance(v, (int, float)) and not isinstance(v, bool): rows.append({"file": p.name, "key": k, "value": round(float(v), 4)})
ev = pd.DataFrame(rows)
print(f"status: {'SMOKE (not a result)' if SMOKE else 'RUN'} | files in {OUT}: {len(list(OUT.glob('*')))}")
print(ev.to_string(index=False) if len(ev) else "no numeric outputs")